In [18]:
pip install openpyxl


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   ---- ---------------------------------- 30.7/250.9 kB 640.0 kB/s eta 0:00:01
   ---------------- ----------------------- 102.4/250.9 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 250.9/250.9 kB 2.2 MB/s eta 0:00:00


In [22]:
pip install cvxpy


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.1/1.1 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------  1.1/1.1 MB 11.5 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 8.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/742.5 kB ? eta -:--:--
   --------------------------------------- 742.5/742.5 kB 23.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/301.9 kB ? eta -:--:--
   ---------------------------------------- 301.9/301.9 kB ? eta 0:00:00
   ---------------------------------------- 0.0/7.4 MB ? eta -:--:--
   ------------ --------------------------- 2.4/7.4 MB 75.6 MB/s eta 0:00:01
   ------------------------ --------------- 4.5/7.4 MB 57.8 MB/s eta 0:00:01
   ------------------------------------- -- 7.0/7.4 MB 56.2 MB/s eta 0:00:01
   ----------------------------------

In [ ]:
pip install pandas 

In [ ]:
pip install numpy

### Python Script for Generating Rebalanced Weights

In [ ]:
import pandas as pd
import numpy as np
import cvxpy as cp

# === Load Excel ===
file_path = "SSX_FINE_6200X_LowMin_Volatility_Portfolio_Fund_Analysis.xlsx"
xls = pd.ExcelFile(file_path)
daily_returns_df = xls.parse("Daily Returns")
daily_returns_df["Date"] = pd.to_datetime(daily_returns_df["Date"])
daily_returns_df.set_index("Date", inplace=True)

# === Mapping Friendly Names to Excel Columns ===
name_map = {
    "TELUS COMMUNICATIONS": "TELUS",
    "VERIZON COMMUNICATIONS": "VERIZON",
    "ROGERS COMMUNICATIONS": "ROGERS",
    "SAUDI TELECOM": "Saudi Telecom",
    "PEPSICO INC": "PEPSICO",
    "THE COCA-COLA COMPANY": "COCA-COLA",
    "JOHNSON & JOHNSON": "JOHNSON&JOHNSON",
    "PROCTER & GAMBLE": "PROCTOR&GAMBLE",
    "ENBRIDGE": "ENBRIDGE",
    "FORTIS INC": "FORTIS",
    "EMERA INC": "EMERA INC",
    "TMX GROUP LIMITED": "TMX GROUP LTD",
    "TORONTO-DOMINION BANK": "TD",
    "ROYAL BANK OF CANADA": "RBC",
    "AL RAJHI BANKING & INVESTMENT CORP.": "Al Rajhi Banking & Investment Corp",
    "INTERMEDIATE CAPITAL GRP": "ICG",
    "GREAT-WEST LIFECO INC": "Great-West Lifeco Inc",
    "MANULIFE": "MANULIFE",
    "SUN LIFE FINANCIAL": "SUNLIFE",
    "RESMED INC": "RESMED",
    "NEWMONT CORP": "NEWMONT",
    "BARRICK GOLD CORP": "BARRICK GOLD",
    "FREEPORT MCMORAN B": "FREEPORT MCMORAN",
    "ANGLO AMERICAN PLC": "ANGLO AMERICAN PLC",
    "WHEATON PRECIOUS METALS": "WHEATON PRECIOUS",
    "NUCOR CORP": "NUCOR CORP",
    "RIYADH CEMENT CO.": "Riyadh Cement Co.",
    "CANADIAN PACIFIC KANSAS CITY LTD": "CANADIAN PACIFIC KANSAS CITY LTD",
    "CANADIAN NATL RAILWAY CO": "CANADIAN NATL RAILWAY CO",
    "WEIR GROUP": "WEIR GROUP",
    "WASTE CONNECTIONS INC": "WASTE CONNECTIONS",
    "U.S. TREASURY BOND": "US Treasury Bond",
    "CVS HEALTH CORP BOND": "CVS Health Corp Bond",
    "CANADIAN TREASURY BONDS": "Canadian Treasury Bond"
}

# === Final Quarter Static Weights ===
final_quarter_weights = {name_map[k]: v for k, v in {
    "TELUS COMMUNICATIONS": 0.0175, "VERIZON COMMUNICATIONS": 0.0175,
    "ROGERS COMMUNICATIONS": 0.0175, "SAUDI TELECOM": 0.0175,
    "PEPSICO INC": 0.02, "THE COCA-COLA COMPANY": 0.02, "JOHNSON & JOHNSON": 0.02, "PROCTER & GAMBLE": 0.02,
    "ENBRIDGE": 0.0433, "FORTIS INC": 0.0434, "EMERA INC": 0.0433,
    "TMX GROUP LIMITED": 0.0375, "TORONTO-DOMINION BANK": 0.0375, "ROYAL BANK OF CANADA": 0.0375,
    "AL RAJHI BANKING & INVESTMENT CORP.": 0.0375, "INTERMEDIATE CAPITAL GRP": 0.0375,
    "GREAT-WEST LIFECO INC": 0.0375, "MANULIFE": 0.0375, "SUN LIFE FINANCIAL": 0.0375,
    "RESMED INC": 0.01, "NEWMONT CORP": 0.022, "BARRICK GOLD CORP": 0.0216,
    "FREEPORT MCMORAN B": 0.0216, "ANGLO AMERICAN PLC": 0.0216, "WHEATON PRECIOUS METALS": 0.0216,
    "NUCOR CORP": 0.0216, "RIYADH CEMENT CO.": 0.026, "CANADIAN PACIFIC KANSAS CITY LTD": 0.026,
    "CANADIAN NATL RAILWAY CO": 0.026, "WEIR GROUP": 0.026, "WASTE CONNECTIONS INC": 0.026,
    "U.S. TREASURY BOND": 0.05, "CVS HEALTH CORP BOND": 0.05, "CANADIAN TREASURY BONDS": 0.05
}.items()}

# === Sector Info ===
sector_weights = {
    "Communications": 0.07, "Consumer Staples": 0.08, "Energy": 0.13,
    "Finance": 0.30, "Medical": 0.01, "Basic Materials": 0.13,
    "Industrial": 0.13, "Bonds": 0.15
}

sector_to_securities = {
    "Communications": ["TELUS", "VERIZON", "ROGERS", "Saudi Telecom"],
    "Consumer Staples": ["PEPSICO", "COCA-COLA", "JOHNSON&JOHNSON", "PROCTOR&GAMBLE"],
    "Energy": ["ENBRIDGE", "FORTIS", "EMERA INC"],
    "Finance": ["TMX GROUP LTD", "TD", "RBC", "Al Rajhi Banking & Investment Corp",
                "ICG", "Great-West Lifeco Inc", "MANULIFE", "SUNLIFE"],
    "Medical": ["RESMED"],
    "Basic Materials": ["NEWMONT", "BARRICK GOLD", "FREEPORT MCMORAN", "ANGLO AMERICAN PLC",
                        "WHEATON PRECIOUS", "NUCOR CORP"],
    "Industrial": ["Riyadh Cement Co.", "CANADIAN PACIFIC KANSAS CITY LTD", "CANADIAN NATL RAILWAY CO",
                   "WEIR GROUP", "WASTE CONNECTIONS"],
    "Bonds": ["US Treasury Bond", "CVS Health Corp Bond", "Canadian Treasury Bond"]
}

# === Prepare Returns ===
mapped_securities = list(final_quarter_weights.keys())
returns = daily_returns_df[mapped_securities].copy()
returns.index = daily_returns_df.index

# === Define Quarters ===
quarters = returns.resample("Q").apply(lambda x: x.index[-1]).index
quarter_labels = [str(q.date()) for q in quarters]

# === Jagannathan & Ma Optimization (with clip) ===
def min_var_portfolio(cov_matrix, sector_weight):
    n = cov_matrix.shape[0]
    w = cp.Variable(n)
    prob = cp.Problem(cp.Minimize(cp.quad_form(w, cov_matrix)), [cp.sum(w) == sector_weight, w >= 0])
    prob.solve()
    if w.value is not None:
        return np.clip(w.value, 0, None)  # ✅ Enforce no negative weights
    else:
        return np.full(n, sector_weight / n)

# === Rebalance Portfolio Quarterly ===
rebalanced_df = pd.DataFrame(index=mapped_securities, columns=quarter_labels)

for qtr in quarters:
    label = str(qtr.date())

    if qtr == quarters[-1]:
        for sec in mapped_securities:
            rebalanced_df.loc[sec, label] = final_quarter_weights[sec]
        continue

    q_start = qtr - pd.offsets.QuarterEnd(1) + pd.Timedelta(days=1)
    q_end = qtr
    q_returns = returns.loc[q_start:q_end].fillna(method="ffill").fillna(method="bfill")

    weights = {}

    for sector, sec_list in sector_to_securities.items():
        valid_secs = [s for s in sec_list if s in q_returns.columns]
        if not valid_secs:
            continue
        cov_matrix = q_returns[valid_secs].cov().values
        sector_wt = sector_weights[sector]
        sector_weights_array = min_var_portfolio(cov_matrix, sector_wt)
        for i, sec in enumerate(valid_secs):
            weights[sec] = sector_weights_array[i]

    for sec in mapped_securities:
        rebalanced_df.loc[sec, label] = weights.get(sec, 0.0)

# === Save to CSV ===
output_path = "LowMin_Volatility_Portfolio_Rebalanced_Weights.csv"
rebalanced_df = rebalanced_df.astype(float)
rebalanced_df.to_csv(output_path)
print(f"Rebalanced weights saved (no negative weights): {output_path}")


FileNotFoundError: [Errno 2] No such file or directory: 'LowMin_Volatility_Portfolio_Analysis_Funds_Comparison.xlsx'